### Filtering transactions

Filtering transactions by removing transactions that already contain a known verb pattern (existing in vp_data3) and modifying the resulting transactions further by removing specific syntactic relations. As an extra step, filtered transactions will be further divided into transactions containing a match to an existing negation pattern (in negations database) and transactions not containing a match to a negation pattern.

Functions `create_filtered_transaction_row()` and `create_filtered_transaction_head()` will be used to create a new filtered transation tables. Function `index_difference()` will be used to find indexes of transactions that should be kept. Functions `remove_deprel_from_transaction_row()` and `remove_aux_verbs()` will be used for further transaction modification.

In [1]:
import sys
sys.path.append('../../../common_code')

In [6]:
import sqlite3
from paths import PATH_ROOT
from db_operations.verb_transactions.transactions_filtering import *
from db_operations.verb_transactions.transaction_modifiers import *

In [7]:
#**Input parameters**

vp_data3 = "C:/Users/liivas/Documents/Töö/verbirektisoonid/vp_data3.db"
v32_data = "C:/Users/liivas/Documents/Töö/verbirektisoonid/v32_data.db"
negations = "C:/Users/liivas/Documents/Töö/verbirektisoonid/db_operations/verb_negations/negations.db"
v32_data_filtered = "v32_data_filtered.db"
v32_data_filtered_neg = "v32_data_filtered_neg.db"
v32_data_filtered_no_neg = "v32_data_filtered_no_neg.db"

# **Data processing**
#connecting to verb patterns database (vp_data3)
con = sqlite3.connect(vp_data3)
cur = con.cursor()

#attaching transactions database (v32)
cur.execute(f'ATTACH DATABASE "{v32_data}" AS v32')

#attaching negation patterns database
cur.execute(f'ATTACH DATABASE "{negations}" AS neg')

#attaching new database for transactions not containing a match to existing verb patterns
cur.execute(f'ATTACH DATABASE "{v32_data_filtered}" AS filtered')

#attaching new database for filtered transactions containing a match to existing negation patterns
cur.execute(f'ATTACH DATABASE "{v32_data_filtered_neg}" AS filtered_neg')

#attaching new database for filtered transactions not containing a match to existing negation patterns
cur.execute(f'ATTACH DATABASE "{v32_data_filtered_no_neg}" AS filtered_no_neg')

# **Filtering transactions**
#creating index table
index_difference(cur,
                 index_tbl_1='verb_matches', # verbs matching verbs in vp_data3 patterns
                 id_col_1='head_id',
                 index_tbl_2='verb_phrase_matches', # phrases containing full match to a pattern in vp_data3
                 id_col_2='head_id',
                 output_tbl='filtered.filtered_head_ids')

#filtering transaction_row
create_filtered_transaction_row(cur,
                                index_tbl='filtered.filtered_head_ids',
                                id_col='head_id',
                                transaction_row='v32.transaction_row',
                                output_tr_row='filtered.transaction_row')

#filtering transaction_head
create_filtered_transaction_head(cur,
                                 index_tbl='filtered.filtered_head_ids',
                                 id_col='head_id',
                                 transaction_head='v32.transaction_head',
                                 output_tr_head='filtered.transaction_head')

# further modifications in filtered transaction_row
remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='nsubj')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='advmod')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='advcl')
remove_deprel_from_transaction_row(cur,
                                   transaction_row='filtered.transaction_row',
                                   deprel='csubj')
remove_aux_verbs(cur,
                 transaction_row='filtered.transaction_row')

In [8]:
# **Filtered transactions containing known negation patterns**
create_filtered_transaction_row(cur,
                                'neg.neg_phrase_matches', # matches to negation patterns
                                'head_id',
                                'filtered.transaction_row',
                                'filtered_neg.transaction_row'
                               )

create_filtered_transaction_head(cur,
                                'neg.neg_phrase_matches',
                                'head_id',
                                'filtered.transaction_head',
                                'filtered_neg.transaction_head')

In [9]:
# **Filtered transactions not containing known negation patterns**
index_difference(cur,
                'filtered.transaction_head', # ID-s from filtered transaction_head
                'id',
                'filtered_neg.transaction_head', # ID-s from filtered negations transaction_head
                'id',
                'filtered_no_neg.filtered_head_ids')

create_filtered_transaction_row(cur,
                                'filtered_no_neg.filtered_head_ids',
                                'head_id',
                                'filtered.transaction_row',
                                'filtered_no_neg.transaction_row')

create_filtered_transaction_head(cur,
                                'filtered_no_neg.filtered_head_ids',
                                'head_id',
                                'filtered.transaction_head',
                                'filtered_no_neg.transaction_head')

con.close()

In [ ]:
# **Alternative way to create filtered transaction_head and transaction_row tables**

#from common_code.db_operations.verb_transactions.filter_verb_transaction_tables import *

#filter_verb_transaction_tables(
#    conn=con,
#    source_schema='v32',
#    transaction_head='transaction_head',
#    transaction_row='transaction_row',
#    target_schema='filtered',
#    new_transaction_head='transaction_head',
#    new_transaction_row='transaction_row',
#    ids_schema='filtered',
#    ids_table='filtered_head_ids',
#    ids_column='head_id',
#    delete_if_exists= True,
#    copy_indexes=True,
#    verbose= True,
#)

NB! Andmebaasi *v32_data_filtered_no_neg.db* tabelitesse *transaction_head* ja *transaction_row* on mõned eitust sisaldavad transaktsioonid alles jäänud. Täpne põhjus on selgumisel, aga näib, et paljudel juhtudel puudub *transaction_head* tabelis olevatel neg verbidel fraasisisu *transaction_row* tabelis (ehk need verbid on üksi, võimalik, et varasema deprelite filtreerimise tagajärjel).